In [1]:
import pandas as pd 
import pickle
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import  train_test_split, GridSearchCV
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from scikeras.wrappers import KerasClassifier

data = pd.read_csv("Churn_Modelling.csv")

data = data.drop(["RowNumber", "CustomerId", "Surname"], axis=1)

label_encode_gender = LabelEncoder()
data["Gender"] = label_encode_gender.fit_transform(data["Gender"])

one_hot_code_Geography = OneHotEncoder()
endcoded_Geography = one_hot_code_Geography.fit_transform(data[["Geography"]])

Geography_df = pd.DataFrame(endcoded_Geography.toarray(), columns=one_hot_code_Geography.get_feature_names_out(["Geography"]))
data=pd.concat([data.drop('Geography',axis=1), Geography_df], axis=1)

with open('label_encode_gender.pkl', 'wb') as file:
    pickle.dump(label_encode_gender, file)

with open('one_hot_code_Geography.pkl', 'wb') as file:
    pickle.dump(one_hot_code_Geography, file)


    ## we divide the data set into independent and dependent variables
X=data.drop('Exited', axis=1) # independent
Y=data['Exited'] # dependent

## we divide the data into training and testing sets
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)


with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)


## define a fn to craete model and try different params

def create_model(neurons=32, layers=1):
    model = Sequential()

    model.add(Dense(neurons, activation='relu', input_shape=(X_train.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neurons, activation='relu'))

    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss="binary_crossentropy", metrics=['accuracy'])

    return model


model = KerasClassifier(build_fn=create_model, epochs=100, batch_size=10, verbose=2)


params_grid = {
    'model__neurons': [16, 32, 64],
    'model__layers': [1, 2, 3],
    'batch_size': [16, 32],
    'epochs': [50, 100]
}

grid = GridSearchCV(estimator=model,  scoring='accuracy', param_grid=params_grid, n_jobs=-1, cv=3)

grid_result = grid.fit(X_train, Y_train)

print(f"best_score={grid_result.best_score_} & best_params={grid_result.best_params_}")


c:\Akash Tiwari\Gen Ai\venv\Lib\site-packages\scikeras\wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)




Epoch 1/100


500/500 - 8s - loss: 0.5086 - accuracy: 0.7800 - 8s/epoch - 15ms/step
Epoch 2/100
500/500 - 3s - loss: 0.4294 - accuracy: 0.8163 - 3s/epoch - 5ms/step
Epoch 3/100
500/500 - 3s - loss: 0.4111 - accuracy: 0.8261 - 3s/epoch - 5ms/step
Epoch 4/100
500/500 - 3s - loss: 0.3946 - accuracy: 0.8336 - 3s/epoch - 6ms/step
Epoch 5/100
500/500 - 2s - loss: 0.3804 - accuracy: 0.8416 - 2s/epoch - 4ms/step
Epoch 6/100
500/500 - 2s - loss: 0.3679 - accuracy: 0.8486 - 2s/epoch - 4ms/step
Epoch 7/100
500/500 - 3s - loss: 0.3596 - accuracy: 0.8530 - 3s/epoch - 5ms/step
Epoch 8/100
500/500 - 2s - loss: 0.3542 - accuracy: 0.8554 - 2s/epoch - 5ms/step
Epoch 9/100
500/500 - 2s - loss: 0.3504 - accuracy: 0.8575 - 2s/epoch - 4ms/step
Epoch 10/100
500/500 - 2s - loss: 0.3473 - accuracy: 0.8583 - 2s/epoch - 4ms/step
Epoch 11/100
500/500 - 2s - loss: 0.3459 - accuracy: 0.8584 - 2s/epoch - 4ms/step
Epoch 12/100
500/500 - 2s - loss: 0.3443 - accuracy: 0.8584 - 2s/epoch - 4ms/step
Epoch 13/100
500/500

In [2]:
best_model = grid_result.best_estimator_      # already trained
y_pred = best_model.predict(X_test)

from sklearn.metrics import accuracy_score
print(accuracy_score(Y_test, y_pred))

125/125 - 3s - 3s/epoch - 21ms/step
0.86


In [3]:
from sklearn.model_selection import RandomizedSearchCV

search = RandomizedSearchCV(
    estimator=model,
    param_distributions=params_grid,
    n_iter=10,
    scoring='accuracy',
    cv=3,
    n_jobs=1,
    verbose=2,
    random_state=42
)

result = search.fit(X_train, Y_train)
print(f"best_score={result.best_score_} & best_params={result.best_params_}")

Fitting 3 folds for each of 10 candidates, totalling 30 fits


c:\Akash Tiwari\Gen Ai\venv\Lib\site-packages\scikeras\wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


Epoch 1/100
167/167 - 4s - loss: 0.4547 - accuracy: 0.8005 - 4s/epoch - 21ms/step
Epoch 2/100
167/167 - 1s - loss: 0.3745 - accuracy: 0.8442 - 513ms/epoch - 3ms/step
Epoch 3/100
167/167 - 1s - loss: 0.3489 - accuracy: 0.8539 - 507ms/epoch - 3ms/step
Epoch 4/100
167/167 - 1s - loss: 0.3380 - accuracy: 0.8560 - 505ms/epoch - 3ms/step
Epoch 5/100
167/167 - 1s - loss: 0.3328 - accuracy: 0.8597 - 512ms/epoch - 3ms/step
Epoch 6/100
167/167 - 1s - loss: 0.3282 - accuracy: 0.8611 - 505ms/epoch - 3ms/step
Epoch 7/100
167/167 - 1s - loss: 0.3219 - accuracy: 0.8620 - 650ms/epoch - 4ms/step
Epoch 8/100
167/167 - 1s - loss: 0.3168 - accuracy: 0.8652 - 583ms/epoch - 3ms/step
Epoch 9/100
167/167 - 0s - loss: 0.3132 - accuracy: 0.8691 - 455ms/epoch - 3ms/step
Epoch 10/100
167/167 - 0s - loss: 0.3107 - accuracy: 0.8669 - 493ms/epoch - 3ms/step
Epoch 11/100
167/167 - 0s - loss: 0.3030 - accuracy: 0.8686 - 468ms/epoch - 3ms/step
Epoch 12/100
167/167 - 0s - loss: 0.3008 - accuracy: 0.8693 - 450ms/epoch - 

c:\Akash Tiwari\Gen Ai\venv\Lib\site-packages\scikeras\wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


Epoch 1/100
167/167 - 3s - loss: 0.4757 - accuracy: 0.7851 - 3s/epoch - 18ms/step
Epoch 2/100
167/167 - 0s - loss: 0.3849 - accuracy: 0.8432 - 492ms/epoch - 3ms/step
Epoch 3/100
167/167 - 0s - loss: 0.3520 - accuracy: 0.8573 - 498ms/epoch - 3ms/step
Epoch 4/100
167/167 - 0s - loss: 0.3433 - accuracy: 0.8609 - 482ms/epoch - 3ms/step
Epoch 5/100
167/167 - 0s - loss: 0.3346 - accuracy: 0.8616 - 492ms/epoch - 3ms/step
Epoch 6/100
167/167 - 0s - loss: 0.3311 - accuracy: 0.8652 - 483ms/epoch - 3ms/step
Epoch 7/100
167/167 - 0s - loss: 0.3251 - accuracy: 0.8626 - 480ms/epoch - 3ms/step
Epoch 8/100
167/167 - 0s - loss: 0.3240 - accuracy: 0.8659 - 472ms/epoch - 3ms/step
Epoch 9/100
167/167 - 0s - loss: 0.3175 - accuracy: 0.8652 - 473ms/epoch - 3ms/step
Epoch 10/100
167/167 - 0s - loss: 0.3148 - accuracy: 0.8682 - 474ms/epoch - 3ms/step
Epoch 11/100
167/167 - 1s - loss: 0.3089 - accuracy: 0.8689 - 618ms/epoch - 4ms/step
Epoch 12/100
167/167 - 0s - loss: 0.3094 - accuracy: 0.8732 - 473ms/epoch - 

c:\Akash Tiwari\Gen Ai\venv\Lib\site-packages\scikeras\wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


... (53 outputs left out)

c:\Akash Tiwari\Gen Ai\venv\Lib\site-packages\scikeras\wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


Epoch 1/100
334/334 - 3s - loss: 0.4303 - accuracy: 0.8189 - 3s/epoch - 10ms/step
Epoch 2/100
334/334 - 1s - loss: 0.3536 - accuracy: 0.8536 - 927ms/epoch - 3ms/step
Epoch 3/100
334/334 - 1s - loss: 0.3350 - accuracy: 0.8620 - 933ms/epoch - 3ms/step
Epoch 4/100
334/334 - 1s - loss: 0.3283 - accuracy: 0.8688 - 933ms/epoch - 3ms/step
Epoch 5/100
334/334 - 1s - loss: 0.3230 - accuracy: 0.8675 - 948ms/epoch - 3ms/step
Epoch 6/100
334/334 - 1s - loss: 0.3172 - accuracy: 0.8699 - 947ms/epoch - 3ms/step
Epoch 7/100
334/334 - 1s - loss: 0.3134 - accuracy: 0.8673 - 945ms/epoch - 3ms/step
Epoch 8/100
334/334 - 1s - loss: 0.3075 - accuracy: 0.8735 - 946ms/epoch - 3ms/step
Epoch 9/100
334/334 - 1s - loss: 0.3022 - accuracy: 0.8757 - 944ms/epoch - 3ms/step
Epoch 10/100
334/334 - 1s - loss: 0.3000 - accuracy: 0.8723 - 947ms/epoch - 3ms/step
Epoch 11/100
334/334 - 1s - loss: 0.2970 - accuracy: 0.8783 - 922ms/epoch - 3ms/step
Epoch 12/100
334/334 - 1s - loss: 0.2946 - accuracy: 0.8795 - 1s/epoch - 3ms

c:\Akash Tiwari\Gen Ai\venv\Lib\site-packages\scikeras\wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


Epoch 1/50
250/250 - 2s - loss: 0.5377 - accuracy: 0.7533 - 2s/epoch - 9ms/step
Epoch 2/50
250/250 - 1s - loss: 0.4440 - accuracy: 0.8124 - 604ms/epoch - 2ms/step
Epoch 3/50
250/250 - 1s - loss: 0.4177 - accuracy: 0.8204 - 592ms/epoch - 2ms/step
Epoch 4/50
250/250 - 1s - loss: 0.3971 - accuracy: 0.8336 - 596ms/epoch - 2ms/step
Epoch 5/50
250/250 - 1s - loss: 0.3793 - accuracy: 0.8422 - 586ms/epoch - 2ms/step
Epoch 6/50
250/250 - 1s - loss: 0.3665 - accuracy: 0.8487 - 588ms/epoch - 2ms/step
Epoch 7/50
250/250 - 1s - loss: 0.3573 - accuracy: 0.8521 - 598ms/epoch - 2ms/step
Epoch 8/50
250/250 - 1s - loss: 0.3521 - accuracy: 0.8541 - 598ms/epoch - 2ms/step
Epoch 9/50
250/250 - 1s - loss: 0.3486 - accuracy: 0.8559 - 603ms/epoch - 2ms/step
Epoch 10/50
250/250 - 1s - loss: 0.3458 - accuracy: 0.8572 - 729ms/epoch - 3ms/step
Epoch 11/50
250/250 - 1s - loss: 0.3451 - accuracy: 0.8546 - 610ms/epoch - 2ms/step
Epoch 12/50
250/250 - 1s - loss: 0.3432 - accuracy: 0.8569 - 601ms/epoch - 2ms/step
Epoc